<a href="https://colab.research.google.com/github/RiceD2KLab/SeeBird/blob/main/WaterbirdDetectionApplication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Waterbird AI Detection Application

This notebook lets you run the various waterbird detection AI models created by the Rice D2K lab in partnership with Houston Audubon. We designed it to be used with Google Colab.

## Installation and setup
Run the code block below to setup the Colab instance and install necessary packages.

* **Make sure you're using a GPU! "Runtime > Change Runtime Type"**
* **This will take ~2-5 minutes to run!**

In [ ]:
import os
import sys
import csv
import zipfile
import numpy as np
import torch
import torchvision
import torchvision.ops as ops
from torchvision.utils import draw_bounding_boxes, make_grid
from torchvision.transforms import v2
import matplotlib.pyplot as plt
import seaborn as sns
import re
import json
from PIL import Image, ImageDraw, ImageFont
from google.colab import files as colabfiles
from ipyfilechooser import FileChooser
import ipywidgets as widgets
from IPython.display import display, clear_output
from collections import Counter
import gdown
!pip install -q ultralytics
from ultralytics import YOLO
import pandas as pd
from tqdm.notebook import tqdm
!pip install -q rasterio
import rasterio
Image.MAX_IMAGE_PIXELS = None

TORCH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {TORCH_DEVICE}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 125.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 103.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultraly

In [ ]:
def tile_img(image_path, tile_size=800, overlap=200, zip_output=False, output_widget=None, overwrite_tiles=False):
    """
    Crops an image into square tiles of specified size with overlap and saves them to an output folder.

    Parameters:
    - image_path (str): Path to the input image file.
    - tile_size (int, optional): Size of each crop tile (width and height in pixels). Default is 640.
    - overlap (int, optional): Overlap between tiles in pixels. Default is 290.
    - zip_output (bool, optional): Whether to zip the tiled images folder. Default is False.
    - output_widget (widgets.Output, optional): Output widget to display messages.
    - overwrite_tiles (bool, optional): Whether to overwrite the existing tiled images folder. Default is False.

    Returns:
    - None. Saves each tile as a separate image file in the specified output folder.
    """
    # Open the image
    try:
        img = Image.open(image_path)
        img_width, img_height = img.size
    except FileNotFoundError:
        if output_widget:
            with output_widget:
                print(f"Error: Image file not found at {image_path}")
        return
    except Exception as e:
        if output_widget:
            with output_widget:
                print(f"Error opening image: {e}")
        return

    # Check if output folder exists and handle overwrite
    base_filename = os.path.splitext(os.path.basename(image_path))[0]
    output_folder = "tiled_" + base_filename
    if os.path.exists(output_folder):
        if overwrite_tiles:
            if output_widget:
                with output_widget:
                    print(f"Overwriting existing tiled images in {output_folder}...")
            import shutil
            shutil.rmtree(output_folder)
        else:
            if output_widget:
                with output_widget:
                    print(f"Found existing tiled images in {output_folder}. Skipping tiling. To overwrite, set overwrite_tiles=True.")
            return output_folder
    # Ensure output folder exists
    os.makedirs(output_folder, exist_ok=True)

    # Calculate step size based on desired overlap
    step_size = tile_size - overlap

    # Create a list of all tile coordinates to iterate over
    tile_coordinates = []
    for top in range(0, img_height, step_size):
        for left in range(0, img_width, step_size):
             # Adjust the step size for the last crops on the right and bottom edges
            current_left = left
            current_top = top
            if current_left + tile_size > img_width:
                current_left = img_width - tile_size  # Shift the crop window left to maintain crop size
            if current_top + tile_size > img_height:
                current_top = img_height - tile_size  # Shift the crop window up to maintain crop size
            tile_coordinates.append((current_left, current_top))

    with tqdm(total=len(tile_coordinates), desc="Tiling Image") as pbar:
        for left, top in tile_coordinates:
            # Define crop boundaries
            right = left + tile_size
            bottom = top + tile_size

            # Tile the image
            tiled_img = img.crop((left, top, right, bottom))

            # Save the cropped tile with the original filename included
            tile_filename = f"{top}_{left}.png"
            tiled_img.save(os.path.join(output_folder, tile_filename))
            pbar.update(1) # Update the progress bar

    if output_widget:
         with output_widget:
             print("Tiling complete.", flush=True) # Explicitly flush output

    if zip_output:
      if output_widget:
          with output_widget:
              print(f"Zipping {output_folder}...", flush=True)
      try:
          with zipfile.ZipFile(f"{output_folder}.zip", 'w', zipfile.ZIP_DEFLATED) as zipf:
              for root, _, files in os.walk(output_folder):
                  for file in files:
                      zipf.write(os.path.join(root, file), os.path.relpath(os.path.join(root, file), os.path.join(output_folder, '..')))
          if output_widget:
              with output_widget:
                  print(f"Zipped {output_folder} to {output_folder}.zip", flush=True)
      except Exception as e:
          if output_widget:
              with output_widget:
                  print(f"Error zipping folder: {e}", flush=True)

    return output_folder

In [ ]:
class TiledImageDataset(torch.utils.data.Dataset):
    def __init__(self, dataset_path, transform=None):
        """
        Args:
            dataset_path (string): Path to a directory or a zip file containing tiled images.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.dataset_path = dataset_path # Store the original path
        self.image_files = []
        self.transform = transform

        self.zip_file = None # Store the zip file object

        if os.path.isdir(dataset_path):
            # If it's a directory, list image files directly
            self.image_files = [os.path.join(dataset_path, f) for f in os.listdir(dataset_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tiff'))]
        elif os.path.isfile(dataset_path) and zipfile.is_zipfile(dataset_path):
             # If it's a zip file, open it and get the list of image files
            try:
                self.zip_file = zipfile.ZipFile(self.dataset_path, 'r')
                self.image_files = [f for f in self.zip_file.namelist() if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tiff'))]
            except zipfile.BadZipFile:
                raise ValueError(f"Invalid zip file: {self.dataset_path}")
            except Exception as e:
                raise RuntimeError(f"Error opening zip file {self.dataset_path}: {e}")
        else:
             raise ValueError(f"Invalid dataset_path: {dataset_path}. Must be a directory or a zip file.")

        self.image_files.sort() # Sort the image files for consistent order

        # Regex to extract coordinates from filename like path/to/image/top_left.png or top_left.png
        self.filename_regex = re.compile(r'(\d+)_(\d+)\.\w+')

    def cleanup(self):
        """Explicitly closes the zip file if it was opened."""
        if self.zip_file:
            self.zip_file.close()
            self.zip_file = None

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        img_info = self.image_files[idx] # This will be a path (for directory) or a filename within the zip (for zip)
        try:
            if self.zip_file:
                # Open image directly from the zip file
                with self.zip_file.open(img_info) as img_file:
                    image = Image.open(img_file).convert("RGB") # Convert to RGB to ensure 3 channels
            else:
                # Open image directly from the directory path
                image = Image.open(img_info).convert("RGB") # Convert to RGB to ensure 3 channels

            if self.transform:
                image = self.transform(image)

        except Exception as e:
            print(f"Error loading image {img_info} from {self.dataset_path}: {e}")
            return None # Return None or handle error appropriately

        # Parse coordinates from filename
        # Need to extract just the filename from the full path for directory case as well
        filename = os.path.basename(img_info)
        match = self.filename_regex.search(filename)
        coordinates = (0, 0) # Default coordinates if parsing fails
        if match:
            try:
                top = int(match.group(1))
                left = int(match.group(2)) # Corrected typo
                coordinates = (top, left)
            except ValueError:
                print(f"Could not parse coordinates from filename: {filename}")

        # Return image and coordinates
        return {'image': image,
                'filename': filename,
                'coordinates': coordinates} # Return the transformed tensor

    def __del__(self):
        """Attempt to close the zip file when the dataset object is deleted."""
        self.cleanup()

In [ ]:
def run_inference(model,
                  dataset,
                  species_map,
                  conf_threshold=0.2,
                  batch_size=16,
                  num_workers=8,
                  output_widget=None,
                  device='cuda'):
  tile_results = []
  if device == 'cpu':
    print("WARNING: YOU ARE USING CPU, WHICH WILL RESULT IN SLOW INFERENCE")

  with tqdm(total=len(dataset), desc="Running inference") as pbar:
    for batch in torch.utils.data.DataLoader(dataset, batch_size=batch_size,
                                             shuffle=False, num_workers=num_workers,
                                             collate_fn=lambda x: x):
      batch_imgs = torch.stack([item['image']
                                for item in batch if item is not None]).to(device)

      # Run
      with torch.no_grad():
        if isinstance(model, YOLO):
          outputs = model(batch_imgs, conf=conf_threshold, verbose=False)
          outputs = [dict(boxes=out.boxes.xyxy.cpu().numpy(),
                          labels=out.boxes.cls.cpu().numpy(),
                          scores=out.boxes.conf.cpu().numpy())
                          for out in outputs]
        else:
          outputs = model(batch_imgs)
          outputs = [dict(boxes=out['boxes'][out['scores'] > conf_threshold].cpu().numpy(),
                          labels=out['labels'][out['scores'] > conf_threshold].cpu().numpy(),
                          scores=out['scores'][out['scores'] > conf_threshold].cpu().numpy())
                          for out in outputs]

      # Update results and progress bar
      tile_results.extend([dict(filename=batch[i]['filename'],
                                top=batch[i]['coordinates'][0],
                                left=batch[i]['coordinates'][1],
                                boxes=outputs[i]['boxes'],
                                scores=outputs[i]['scores'],
                                labels=outputs[i]['labels'],
                                label_ids=[species_map[label] for label in outputs[i]['labels']])
                           for i in range(len(batch))])
      pbar.update(len(batch))
  print("Inference complete.")
  return tile_results

CLASS_COLOR_MAP = {
    'Bird': (255, 0, 0), # Red
    'LAGU-Breeding': (255, 0, 0), # Red
    'BRPE-Breeding': (139, 69, 19), # Brown
    'BRPE-Chick': (255, 105, 180), # Pink
    'Flying': (0, 255, 0),  # Green
    'ROYT': (255, 255, 0),  # Yellow
    'CATE': (128, 0, 128),  # Purple
    'SATE': (0, 255, 255),  # Cyan
    'Tern Spp.': (255, 165, 0), # Orange
    'White Wader': (255, 255, 255),# White
    'REEG': (0, 128, 0),  # Dark Green
    'TRHE': (128, 128, 0),  # Olive
    'GBHE': (0, 0, 255),  # Blue
    'BCNH': (0, 0, 0),  # Black
    'ROSP': (123, 104, 238),  # Medium Slate Blue
    'Other Spp.': (210, 180, 140),  # Tan
}

def plot_example_results(filepaths, results, bbox_width=3, **make_grid_kwargs):
  plot_images = []
  for fpath, result in zip(filepaths, results):
    img = Image.open(fpath).convert('RGB')
    if result['boxes'].shape[0] > 0:
      plot_images.append(draw_bounding_boxes(
          image=v2.functional.pil_to_tensor(img),
          boxes=torch.Tensor(result['boxes']),
          labels=result['label_ids'],
          colors=[CLASS_COLOR_MAP[l_id] if l_id in CLASS_COLOR_MAP else 'red'
                  for l_id in result['label_ids']],
          width=bbox_width)
      )
    # Don't draw bounding boxes if there aren't any!
    else:
      plot_images.append(v2.functional.pil_to_tensor(img))
  plot_images = torchvision.utils.make_grid(plot_images, **make_grid_kwargs)
  display(v2.functional.to_pil_image(plot_images))

In [ ]:
def convert_tile_results_to_pd(results):
  df = dict(
    tile_filename = [],
    tile_top=[],
    tile_left=[],
    x0 = [],
    y0 = [],
    x1 = [],
    y1 = [],
    label = [],
    label_id = [],
    score = [],
  )
  for result in results:
    for i in range(len(result['boxes'])):
      df['tile_filename'].append(result['filename'])
      df['tile_top'].append(result['top'])
      df['tile_left'].append(result['left'])
      df['x0'].append(result['boxes'][i][0])
      df['y0'].append(result['boxes'][i][1])
      df['x1'].append(result['boxes'][i][2])
      df['y1'].append(result['boxes'][i][3])
      df['label'].append(result['labels'][i])
      df['label_id'].append(result['label_ids'][i])
      df['score'].append(result['scores'][i])
  return pd.DataFrame(df)


def combine_tile_results(all_tile_results, iou_threshold=0.6):
  """
  Adjusts bounding boxes from multiple tiles' inference results to align with the original image
  coordinates and applies NMS to remove duplicates across all tiles.

  Parameters:
  - all_tile_results (pd.DataFrame): DataFrame containing tile-level inference results for a single image.
  - iou_threshold (float): IoU threshold for NMS.
  Returns:
  - final_bboxes (Tensor): Adjusted bounding boxes after merging.
  - final_scores (Tensor): Adjusted confidence scores after merging.
  - final_labels (Tensor): Adjusted class labels after merging.
  """
  # Gather all predictions from all tiles
  all_bboxes, all_scores, all_labels = [], [], []
  for _, row in all_tile_results.iterrows():
    # Adjust bounding boxes for this tile
    adjusted_box = [row['x0'] + row['tile_left'],
                    row['y0'] + row['tile_top'],
                    row['x1'] + row['tile_left'],
                    row['y1'] + row['tile_top']]
    all_bboxes.append(adjusted_box)

    # Accumulate scores and labels
    all_scores.append(row['score'])
    all_labels.append(row['label'])

  # Ensure bounding boxes are a 2D tensor
  if all_bboxes:
    all_bboxes = torch.tensor(all_bboxes, dtype=torch.float32)
  else:
    all_bboxes = torch.empty((0, 4), dtype=torch.float32)

  # Ensure scores are a 1D tensor
  if all_scores:
    all_scores = torch.tensor(all_scores, dtype=torch.float32)
  else:
    all_scores = torch.empty(0, dtype=torch.float32)

  # Ensure labels are a 1D tensor
  if all_labels:
    all_labels = torch.tensor(all_labels, dtype=torch.int64)
  else:
    all_labels = torch.empty(0, dtype=torch.int64)

  # Apply NMS on combined adjusted bounding boxes
  if len(all_bboxes) > 0:
    keep_indices = ops.nms(all_bboxes, all_scores, iou_threshold)
    final_bboxes = all_bboxes[keep_indices]
    final_scores = all_scores[keep_indices]
    final_labels = all_labels[keep_indices]
  else:
    final_bboxes = torch.empty((0, 4), dtype=torch.float32)
    final_scores = torch.empty(0, dtype=torch.float32)
    final_labels = torch.empty(0, dtype=torch.int64)

  return final_bboxes, final_scores, final_labels


def create_labelstudio_tasks(labelstudio_local_dir: str,
                             model_name: str,
                             image_filenames: list,
                             image_results: list[dict],
                             raw_img_height=800,
                             raw_img_width=800):
  label_studio_tasks = []
  for filename, results in zip(image_filenames, image_results):
    if len(results['boxes']) == 0:
      continue
    if isinstance(results['boxes'], (np.ndarray, torch.Tensor)):
      results['boxes'] = results['boxes'].tolist()
      results['scores'] = results['scores'].tolist()
    task = {
      "data": {
          "image": os.path.join(f"/data/local-files/?d={labelstudio_local_dir}", filename)
        },
      "predictions": [
          {"model_version": model_name,
          #  "score": np.mean(results['scores']).item(),
           "result": [{
               "id": f"result-{i}",
               "type": "rectanglelabels",
               "from_name": "label",
               "to_name": "image",
               "original_width": raw_img_height,
               "original_height": raw_img_width,
               "image_rotation": 0,
               "value": {
                   "rotation": 0,
                   "x": 100. * results['boxes'][i][0]/raw_img_width,
                   "y": 100. * results['boxes'][i][1]/raw_img_height,
                   "width": 100. * (results['boxes'][i][2]-results['boxes'][i][0])/raw_img_width,
                   "height": 100. * (results['boxes'][i][3]-results['boxes'][i][1])/raw_img_height,
                   "rectanglelabels": [results['label_ids'][i]]
              }
               } for i in range(len(results['boxes']))]
          }]
      }
    label_studio_tasks.append(task)
  return label_studio_tasks

## Application:

Parameters:
*   **Upload Image / File Chooser**: Use the "Upload Image" button to upload a new image file, or the file chooser to select an image file (png, jpg, jpeg) that has already been uploaded or is available in your Colab environment.
*   **Tile Size**: The size (width and height in pixels) of the square tiles the image will be divided into for processing. (Suggestion: 800)
*   **Overlap**: The number of pixels that each adjacent tile will overlap. This helps ensure objects on tile edges are not missed during detection. (Suggestion: 200)
*   **Zip Output**: If checked, the folder containing the tiled images will be zipped after tiling is complete. (Suggestion: Keep on, and download the file after tiling is complete. You can re-upload to save time in the future!)
*   **Overwrite Tile Folder**: If checked, the existing tiled images folder with the same name will be deleted before creating new tiles.
*   **Batch Size**: The number of tiled images that will be processed by the model at the same time during inference. A larger batch size can speed up inference but requires more GPU memory. (Suggestion: 8)
*   **Confidence Threshold**: The minimum confidence score for a detected object to be included in the results. Objects detected with a score below this threshold will be discarded. (Suggestion: 0.2)
*   **Display Inference**: If checked, the application will display example images with the detected bounding boxes and a count plot of the detected species after inference is complete.
*   **Labelstudio Local Dir**: This specifies the local directory path that Label Studio will use to access the images if you import the generated Label Studio tasks. You should replace "path/to/images/" with the actual path where your images will be stored relative to your Label Studio project.



In [ ]:
# @title Load model
MODEL_NAME = "bird_only_fasterrcnn_r50_fpn" # @param ["bird_only_fasterrcnn_r50_fpn", "yolov10"]

MODEL_WEIGHTS_GDRIVE_IDS = {
    "bird_only_fasterrcnn_r50_fpn": "1tgUUEo2W19qtIANGHPVzFfQnQKLZe1GM",
    "yolov10": "1a4TgkF0DeIfI6k0CNculrrkeQZjsPR3D",
}

def load_model(model_name):
  # Download model if it does not exist
  if not os.path.exists(f"/content/{model_name}.pt"):
    gdown.download(id=MODEL_WEIGHTS_GDRIVE_IDS[model_name],
                  output=f"/content/{model_name}.pt",
                  quiet=False)
  # Load appropriate model
  if model_name == "bird_only_fasterrcnn_r50_fpn":
    model = torch.load("/content/bird_only_fasterrcnn_r50_fpn.pt",
                       map_location=TORCH_DEVICE)
    species_map = {1: "Bird"}
  elif model_name == "yolov10":
    model = YOLO(f"/content/yolov10.pt").to(TORCH_DEVICE)
    species_map = model.names
  else:
    RuntimeError(f"Model {model_name} not supported")
  return model, species_map

MODEL, SPECIES_MAP = load_model(MODEL_NAME)

Downloading...
From: https://drive.google.com/uc?id=1a4TgkF0DeIfI6k0CNculrrkeQZjsPR3D
To: /content/yolov10.pt
100%|██████████| 5.76M/5.76M [00:00<00:00, 33.7MB/s]


In [ ]:
for name in SPECIES_MAP.values():
  print(name)

LAGU-Breeding
BRPE-Breeding
BRPE-Chick
Flying
ROYT
CATE
SATE
Tern Spp.
White Wader
REEG
TRHE
GBHE
BCNH
ROSP
Other Spp.


In [ ]:
# @title Run application
def run_application():
  # Setup widgets
  output_widget = widgets.Output()

  upload_button = widgets.Button(description="Upload Image")
  file_chooser = FileChooser('.') # FileChooser to select existing files
  file_chooser.filter_pattern = ['*.png', '*.jpg', '*.jpeg'] # Filter for image files
  tile_size_input = widgets.IntText(description="Tile Size:", value=800)
  overlap_input = widgets.IntText(description="Overlap:", value=200)
  batch_size_input = widgets.IntSlider(description="Batch Size:",
                                       value=8, min=1, max=128)
  conf_threshold_input = widgets.FloatSlider(description="Confidence Threshold:",
                                             value=0.2, min=0.0, max=1.0, step=0.01)
  display_inference_checkbox = widgets.Checkbox(description="Display Inference", value=True)
  labelstudio_local_dir_input = widgets.Text(description="Labelstudio Local Dir:", value="path/to/images/")
  overwrite_tile_folder_checkbox = widgets.Checkbox(description="Overwrite Tile Folder", value=False)
  zip_checkbox = widgets.Checkbox(description="Zip Output", value=True)

  # Create buttons with styles
  run_button = widgets.Button(description="Run!", style=widgets.ButtonStyle(button_color='darkgreen'))

  uploaded_image_path = None # Keep this to track if an image was uploaded

  def on_upload_button_clicked(_):
      nonlocal uploaded_image_path
      with output_widget:
          clear_output()
          uploaded = colabfiles.upload()
          for name, data in uploaded.items():
              with open(name, 'wb') as f:
                  f.write(data)
              print(f'Uploaded {name}')
              uploaded_image_path = name # Set uploaded path
              # Do not clear file_chooser.selected here
              break # Assuming only one file is uploaded

  def on_file_chooser_select(_):
      # This function is not strictly necessary for the logic in run_button_clicked,
      # but can be used for displaying the selected file name if desired.
      pass

  def on_run_button_clicked(_):
      with output_widget:
          clear_output()
          # Prioritize the file selected in the file chooser
          image_to_tile = file_chooser.selected if file_chooser.selected else uploaded_image_path

          if image_to_tile:
              tile_size = tile_size_input.value
              overlap = overlap_input.value
              zip_output = zip_checkbox.value
              overwrite_tile_folder = overwrite_tile_folder_checkbox.value
              print(f"Running tiling for {image_to_tile} with tile size {tile_size},",
                    f"overlap {overlap}, zip output: {zip_output}, delete old tiles: {overwrite_tile_folder}")

              # 1. Run tiling directly
              tile_output_folder = tile_img(image_to_tile, tile_size, overlap,
                                            zip_output, output_widget, overwrite_tile_folder) # Pass output_widget

              # 2. Setup dataset and dataloader
              dataset = TiledImageDataset(tile_output_folder,
                                          transform=v2.Compose([v2.ToTensor(),
                                                                v2.ToDtype(torch.float32, scale=True)]))

              # 3. Run inference
              conf_threshold = conf_threshold_input.value
              batch_size = batch_size_input.value
              plot_inference = display_inference_checkbox.value
              print(f"Running inference for {len(dataset)} image tiles, plot inference examples: {plot_inference}")
              results = run_inference(MODEL,
                                      dataset,
                                      SPECIES_MAP,
                                      conf_threshold=conf_threshold,
                                      batch_size=batch_size,
                                      num_workers=2,
                                      device=TORCH_DEVICE,
                                      output_widget=output_widget)
              # if plot_inference:
              #   filepaths = [os.path.join(dataset.dataset_path, result['filename'])
              #                for result in results]
              #   plot_example_results(filepaths[:12], results[:12],
              #                        bbox_width=4,
              #                        nrow=4)

              # 4. Save results as CSVs
              # Create results directory
              os.makedirs(f"results/{MODEL_NAME}", exist_ok=True)
              # Create pandas df with original tiled results
              results_df = convert_tile_results_to_pd(results)
              # Save original tiled results as csv
              results_df.to_csv(f"results/{MODEL_NAME}/tiled_results_{os.path.basename(image_to_tile).split('.')[0]}.csv", index=False)
              print("Saved tiled results to:", f"results/{MODEL_NAME}/tiled_results_{os.path.basename(image_to_tile).split('.')[0]}.csv")
              # Combine tiled results
              final_bboxes, final_scores, final_labels = combine_tile_results(results_df)
              combined_results_df = pd.DataFrame({
                  'x0': final_bboxes[:, 0].tolist(),
                  'y0': final_bboxes[:, 1].tolist(),
                  'x1': final_bboxes[:, 2].tolist(),
                  'y1': final_bboxes[:, 3].tolist(),
                  'label': final_labels.tolist(),
                  'label_id': [SPECIES_MAP[l_id] for l_id in final_labels.tolist()],
                  'score': final_scores.tolist()
              })
              combined_results_df.to_csv(f"results/{MODEL_NAME}/{os.path.basename(image_to_tile).split('.')[0]}.csv", index=False)
              print("Saved combined results to: ", f"results/{MODEL_NAME}/{os.path.basename(image_to_tile).split('.')[0]}.csv")
              if plot_inference:
                plot_example_results(
                    [image_to_tile],
                    [dict(boxes=final_bboxes.numpy(),
                          label_ids=[SPECIES_MAP[l_id] for l_id in final_labels.numpy()])],
                    bbox_width=10,
                    nrow=1)
                plt.figure()
                sns.countplot(combined_results_df, y="label_id")
                plt.grid(axis='x')
                plt.show()

              # 5. Create and save label studio tasks
              labelstudio_local_dir = labelstudio_local_dir_input.value
              # Tiled tasks
              tiled_labelstudio_tasks = create_labelstudio_tasks(
                  labelstudio_local_dir,
                  MODEL_NAME,
                  [os.path.join(tile_output_folder, res['filename']) for res in results],
                  results,
                  800, 800)
              with open(f"results/{MODEL_NAME}/tiled_tasks_{os.path.basename(image_to_tile).split('.')[0]}.json", 'w') as f:
                json.dump(tiled_labelstudio_tasks, f)
              print("Saved label studio tasks for tiled images at",
                    f"results/{MODEL_NAME}/tiled_tasks_{os.path.basename(image_to_tile).split('.')[0]}.json")
              # Combined tasks
              orig_img_width, orig_img_height = Image.open(image_to_tile).size
              combined_labelstudio_tasks = create_labelstudio_tasks(
                  labelstudio_local_dir,
                  MODEL_NAME,
                  [os.path.basename(image_to_tile)],
                  [dict(boxes=final_bboxes.tolist(),
                        scores=final_scores.tolist(),
                        label_ids=[SPECIES_MAP[l_id] for l_id in final_labels.tolist()])],
                  orig_img_height, orig_img_width)
              with open(f"results/{MODEL_NAME}/task_{os.path.basename(image_to_tile).split('.')[0]}.json", 'w') as f:
                json.dump(combined_labelstudio_tasks, f)
              print("Saved label studio tasks for entire image at",
                    f"results/{MODEL_NAME}/task_{os.path.basename(image_to_tile).split('.')[0]}.json")
          else:
              print("Please upload or select an image first.")

  upload_button.on_click(on_upload_button_clicked)
  file_chooser.observe(on_file_chooser_select, names='selected') # Still observe for potential future use or debugging
  run_button.on_click(on_run_button_clicked)

  # Arrange buttons side by side (only run button now)
  button_box = widgets.HBox([run_button]) # Only include run_button

  display(upload_button,
          file_chooser, tile_size_input, overlap_input, zip_checkbox, overwrite_tile_folder_checkbox,
          batch_size_input, conf_threshold_input, display_inference_checkbox,
          labelstudio_local_dir_input,
          button_box,
          output_widget)

run_application()


Button(description='Upload Image', style=ButtonStyle())

FileChooser(path='/content', filename='', title='', show_hidden=False, select_desc='Select', change_desc='Chan…

IntText(value=800, description='Tile Size:')

IntText(value=200, description='Overlap:')

Checkbox(value=True, description='Zip Output')

...

IntSlider(value=8, description='Batch Size:', max=128, min=1)

FloatSlider(value=0.2, description='Confidence Threshold:', max=1.0, step=0.01)

Checkbox(value=True, description='Display Inference')

Text(value='path/to/images/', description='Labelstudio Local Dir:')

Output()